# Exercise — Audit & Remediate Catalog Metadata (Customer & Marketing) (Solution)

**Trailhead Provisions** is under a GDPR audit notice. Using the **AWS Glue Data Catalog**, find
where governance controls are missing for the two domains that hold personal data —
**Customer Identity & Loyalty** and **Marketing & Campaigns** — **fix the gaps in the catalog**,
verify with a re-audit, then write a short reconciliation memo. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

created Glue database 'trailhead'
registered s3://123456789012-trailhead-data with Lake Formation
  customer               rows=  310  table-gaps=[]  untagged-cols=['email']
  orders                 rows= 1200  table-gaps=['owner']  untagged-cols=[]
  order_returns          rows=  106  table-gaps=[]  untagged-cols=[]
  supplier               rows=   40  table-gaps=['retention']  untagged-cols=[]
  product                rows=  200  table-gaps=['retention']  untagged-cols=['unit_cost']
  shipment               rows= 1109  table-gaps=['classification']  untagged-cols=[]
  campaign               rows=   20  table-gaps=['owner']  untagged-cols=[]
  campaign_membership    rows=  800  table-gaps=['owner', 'retention']  untagged-cols=['cust_ref']

Done. Glue database 'trailhead' seeded in s3://123456789012-trailhead-data with Lake Formation enforcement enabled.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)


## 1. Run the catalog audit (provided)

In [3]:
audit = gc.catalog_audit()
print('Total gaps across all domains:', len(audit))
audit

Total gaps across all domains: 10


,table,domain,gap_type,detail
0,customer,Customer Identity & Loyalty,untagged_column,email (type TEXT)
1,orders,E-commerce Orders,missing_owner,owner
2,supplier,Inventory & Fulfillment,missing_retention,retention
3,product,Inventory & Fulfillment,missing_retention,retention
4,product,Inventory & Fulfillment,untagged_column,unit_cost (type REAL)
5,shipment,Inventory & Fulfillment,missing_classification,classification
6,campaign,Marketing & Campaigns,missing_owner,owner
7,campaign_membership,Marketing & Campaigns,missing_owner,owner
8,campaign_membership,Marketing & Campaigns,missing_retention,retention
9,campaign_membership,Marketing & Campaigns,untagged_column,cust_ref (type TEXT)


## 2. Narrow to the Customer & Marketing domains
Filter the audit to the two domains you're responsible for, and inspect the column descriptions of `customer` and `campaign_membership` to see what each untagged column actually stores.

In [4]:
my_domains = ["Customer Identity & Loyalty", "Marketing & Campaigns"]
my_gaps = audit[audit["domain"].isin(my_domains)]
my_gaps

,table,domain,gap_type,detail
0,customer,Customer Identity & Loyalty,untagged_column,email (type TEXT)
6,campaign,Marketing & Campaigns,missing_owner,owner
7,campaign_membership,Marketing & Campaigns,missing_owner,owner
8,campaign_membership,Marketing & Campaigns,missing_retention,retention
9,campaign_membership,Marketing & Campaigns,untagged_column,cust_ref (type TEXT)


In [5]:
# Provided: inspect column descriptions so you can see what `customer.email` and `campaign_membership.cust_ref` store.
print("--- customer columns ---")
display(gc.describe("customer"))
print("--- campaign_membership columns ---")
display(gc.describe("campaign_membership"))

--- customer columns ---


,column,type,classification,description
0,customer_id,INTEGER,PUBLIC,surrogate key
1,full_name,TEXT,PII,customer name
2,email,TEXT,(untagged),primary contact email
3,phone,TEXT,PII,contact phone
4,region,TEXT,PUBLIC,service region; drives row policy
5,country,TEXT,PUBLIC,
6,loyalty_tier,TEXT,PUBLIC,
7,marketing_consent,INTEGER,SENSITIVE,GDPR consent (loyalty view)
8,created_ts,TEXT,PUBLIC,


--- campaign_membership columns ---


,column,type,classification,description
0,membership_id,INTEGER,PUBLIC,
1,campaign_id,INTEGER,PUBLIC,
2,cust_ref,TEXT,(untagged),customer EMAIL (marketing's key)
3,segment,TEXT,PUBLIC,
4,mkt_consent,INTEGER,SENSITIVE,marketing consent (marketing view)


## 3. Remediate the gaps in the catalog
Write the missing governance metadata back to Glue with the provided helpers:
- `gc.set_table_metadata(table, owner=..., classification=..., retention=...)`
- `gc.tag_column(table, column, classification)`

Valid platform classification values are **`PII`**, **`SENSITIVE`**, and **`PUBLIC`**. Be sure to classify personal-data columns (and PII-bearing tables) as **`PII`**.

In [6]:
# Fix every gap in the two owned domains.
gc.tag_column("customer", "email", "PII")                       # untagged PII column
gc.set_table_metadata("campaign", owner="marketing@trailhead.example")
gc.set_table_metadata("campaign_membership", owner="marketing@trailhead.example",
                      classification="PII", retention="P3Y")    # ungoverned PII table
gc.tag_column("campaign_membership", "cust_ref", "PII")         # second untagged PII column (customer email)
print("Remediation written to AWS Glue Data Catalog.")

Remediation written to AWS Glue Data Catalog.


## 4. Re-audit to verify (provided)
Confirm your two domains have 0 remaining gaps and that personal-data columns (`customer.email` and `campaign_membership.cust_ref`) carry the `PII` classification.

In [7]:
re_audit = gc.catalog_audit()
remaining = re_audit[re_audit["domain"].isin(my_domains)]
email_cls = gc.catalog.table("customer").column("email").classification
cust_ref_cls = gc.catalog.table("campaign_membership").column("cust_ref").classification
print("Gaps in your domains -> before:", len(my_gaps) if my_gaps is not None else "?",
      "after:", len(remaining))
print(f"Column classifications -> customer.email={email_cls!r}, campaign_membership.cust_ref={cust_ref_cls!r}")
assert len(remaining) == 0, f"Expected 0 remaining gaps in {my_domains}, got {len(remaining)}"
assert email_cls == "PII" and cust_ref_cls == "PII", (
    "Both customer.email and campaign_membership.cust_ref hold personal data and must be tagged 'PII'"
)
remaining

Gaps in your domains -> before: 5 after: 0
Column classifications -> customer.email='PII', campaign_membership.cust_ref='PII'


,table,domain,gap_type,detail


## 5. Review the cross-domain conflicts (provided)

In [8]:
gc.semantic_conflicts()

                  concept                     domain_a                                   field_a               domain_b                                                                         field_b                                                                                                                                                                         impact
0   customer identity key  Customer Identity & Loyalty  customer.customer_id (INTEGER surrogate)  Marketing & Campaigns  campaign_membership.cust_ref (TEXT — 758/800 [94.75%] email strings, 42 'n/a')  758 of 800 (94.75%) campaign memberships key customers by email string (plus 42 unresolvable 'n/a' rows) instead of customer_id; cross-domain resolution must match on email.
1       marketing consent  Customer Identity & Loyalty                customer.marketing_consent  Marketing & Campaigns                                                 campaign_membership.mkt_consent                                  157 of 758 valid-

## 6. Write your reconciliation memo
Replace the cell below with your memo. Address every requirement in `INSTRUCTIONS.md`.

### Reconciliation memo — Customer & Marketing domains

**Highest-severity gaps (GDPR-critical), now remediated in the catalog:**
- **`customer.email` was an untagged PII column.** Email is personal data, but it carried no
  classification — invisible to any PII-driven access or subject-access workflow. I tagged it
  `PII`. This is the first thing the GDPR audit would flag.
- **`campaign_membership.cust_ref` was also untagged PII** — inspecting `gc.describe("campaign_membership")`
  shows its description is `"customer EMAIL (marketing's key)"`, a second ungoverned copy of
  personal data. Tagged `PII`.

**Table-level gaps, now remediated:**
- `campaign` had **no owner** — no accountable steward; set to `marketing@trailhead.example`.
- `campaign_membership` had **no owner and no retention** (and held personal data in `cust_ref`) —
  ungoverned PII with no deletion clock, the worst combination under GDPR; set
  `owner="marketing@trailhead.example"`, `classification="PII"`, and `retention="P3Y"`.

**Cross-domain semantic conflicts affecting these domains:**
- **Customer identity key:** loyalty keys customers on `customer_id` (integer) while marketing
  keys on `cust_ref` (email). No shared key, so customer data can only be reconciled across
  domains by matching on email — fragile and a resolution risk.
- **Marketing consent:** `customer.marketing_consent` (loyalty) and
  `campaign_membership.mkt_consent` (marketing) are two systems of record that disagree. Acting
  on the marketing copy when loyalty has withdrawn consent is a direct GDPR violation.

**Recommended standards:** gate publish on `owner`, `classification` (`PII` / `SENSITIVE` /
`PUBLIC`), and `retention` for every table; apply the platform `PII` tag to personal-data columns
at creation; and designate the loyalty domain as the single consent system of record that
marketing reads rather than copies.

---
## Solution Summary & Common Mistakes to Avoid

### Key points
- **Metadata as an active control surface:** auditing AWS Glue finds where governance controls (`owner`, `classification`, `retention`, and column-level `PII` tags) are absent, and writing those attributes back to Glue closes the compliance gap.
- **Inspecting column descriptions before tagging:** `gc.describe("campaign_membership")` reveals that `cust_ref` stores `"customer EMAIL (marketing's key)"`, so both `customer.email` and `campaign_membership.cust_ref` (plus `campaign_membership` at the table level) must be classified `PII`.
- **Semantic conflicts:** metadata reconciliation also surfaces where two domains define shared concepts differently (`customer_id` vs. `cust_ref` email key, and `customer.marketing_consent` vs. `campaign_membership.mkt_consent`).

### Common Mistakes to Avoid
- **Auditing but not remediating** — the skill is closing the gap in AWS Glue and proving it with a re-audit, not just listing findings.
- **Tagging personal-data columns as `PUBLIC` or `SENSITIVE` instead of `PII`** — `customer.email` and `campaign_membership.cust_ref` hold customer email addresses and must be classified `PII`.
- **Listing the untagged column as merely "missing a tag"** without recognizing it holds PII — the classification gap on a personal-data column is the headline GDPR finding.
- **Auditing all domains** instead of narrowing to the two owned domains (`Customer Identity & Loyalty` and `Marketing & Campaigns`).
- **Treating the consent disagreement as a data-quality bug** rather than a governance conflict requiring a single system of record.